# Validation of the frozen dataset and of the numerical claims in the manuscript

Run this before any submission. It recomputes, from `dataset_frozen_2026-09-12/`, every
number the manuscript and the inclusion argument assert about the sample, and prints
PASS or FAIL against the value currently claimed.

Nothing here modifies the dataset. It only reads.

**What the sections cover**

1. Setup and integrity of the register
2. Sample totals
3. Connectivity, which networks are fitted on a fragment
4. Technology mix and the non steel wheel lines
5. The steel wheel counterfactual
6. Scope, the metropolitan lines and the urban scope sample
7. Table 1 indicators and the East Asian means
8. Cross city shared stations
9. Claim ledger, every asserted number in one table

## 1. Setup and register integrity

In [ ]:
import json, csv, math, collections
from pathlib import Path
import numpy as np
import networkx as nx

FROZEN = Path('.').resolve()
if not (FROZEN / 'inclusion_table_frozen.csv').exists():
    FROZEN = Path('August - revision/dataset_frozen_2026-09-12').resolve()
ROOT = FROZEN.parents[1]
print('frozen dataset:', FROZEN)
print('repo root     :', ROOT)

CHECKS = []
def check(label, got, want, note=''):
    ok = (got == want)
    CHECKS.append(dict(label=label, got=got, want=want, ok=ok, note=note))
    print(('PASS  ' if ok else 'FAIL  ') + label + '  got=' + repr(got) + '  claimed=' + repr(want) + ('  ' + note if note else ''))
    return ok

In [ ]:
rows_all = list(csv.DictReader(open(FROZEN / 'inclusion_table_frozen.csv', encoding='utf-8-sig')))
rows = [r for r in rows_all if not r['verdict'].startswith('exclude')]
excluded = [r for r in rows_all if r['verdict'].startswith('exclude')]

print('register rows   :', len(rows_all))
print('included        :', len(rows))
print('excluded        :', len(excluded))
print('cities          :', len(set(r['city'] for r in rows_all)))
print()
for r in excluded:
    print('  EXCLUDED  %-10s %-20s T1=%-5s T2=%s' % (r['city'], r['line_local'][:20],
          r['T1_uitp_line_criteria'], r['T2_designated_urban_system'][:22]))

check('register rows', len(rows_all), 420)
check('included lines', len(rows), 418)
check('cities', len(set(r['city'] for r in rows_all)), 62)

### 1.1 Which T1 factors are actually evidenced per line

The criterion has four limbs: guidance, traction, right of way and train size. This shows
how much of that the register can demonstrate, which bears on what the paper may claim
about transparency.

In [ ]:
n = len(rows)
for col in ['technology', 'technology_source', 'train_formation', 'capacity_per_train',
            'capacity_class_national', 'legal_class']:
    filled = sum(1 for r in rows if r.get(col, '').strip())
    print('  %-26s %3d / %d  (%3.0f%%)' % (col, filled, n, 100 * filled / n))

print()
missing_cols = [c for c in ['section_endpoints', 'retrieval_date', 'approval_instrument']
                if c not in rows_all[0]]
print('columns reviewer facing text refers to but register lacks:', missing_cols)

## 2. Sample totals

In [ ]:
def load_L(d):
    out = {}
    for f in sorted(Path(d).glob('*-L.json')):
        out[f.name[:-7]] = json.load(open(f, encoding='utf-8'))
    return out

LIN = {
    'frozen v1': load_L(FROZEN / 'L2'),
    'frozen v2': load_L(FROZEN / 'L2_v2'),
    'source v1': load_L(ROOT / 'asia' / 'L2'),
    'source v2': load_L(ROOT / 'asia' / 'L2_v2'),
}
for k, g in LIN.items():
    print('%-11s %2d cities, %5d stations' % (k, len(g), sum(len(d['nodes']) for d in g.values())))

check('v1 pre freeze stations',  sum(len(d['nodes']) for d in LIN['source v1'].values()), 7456)
check('v1 frozen stations',      sum(len(d['nodes']) for d in LIN['frozen v1'].values()), 7400)
check('v2 pre freeze stations',  sum(len(d['nodes']) for d in LIN['source v2'].values()), 7480)
check('v2 frozen stations',      sum(len(d['nodes']) for d in LIN['frozen v2'].values()), 7428)

### 2.1 The seven corrections, city by city

Each row is a correction applied by the freeze. FAIL here means the graph does not match
what the manifest says was done.

In [ ]:
CORRECTIONS = [
    ('Daegu',      91,  88, 'Line 1 truncated at Anshim'),
    ('Xian',      238, 237, 'Xihu Line excluded'),
    ('Seoul',     296, 285, 'Line 7 truncated at Onsu'),
    ('Guangzhou', 309, 292, 'Guangfo Line divided at the boundary'),
    ('Foshan',     64,  56, 'Guangfo Line divided at the boundary'),
    ('Beijing',   417, 401, 'two trams removed'),
    ('Incheon',    68,  68, 'unchanged, retains the Line 7 section'),
    ('Changchun', 111, 111, 'unchanged, Line 3 retained'),
]
for city, before, after, why in CORRECTIONS:
    b = len(LIN['source v1'][city]['nodes'])
    a = len(LIN['frozen v1'][city]['nodes'])
    ok = (b == before and a == after)
    print(('PASS  ' if ok else 'FAIL  ') + '%-10s %3d -> %3d  (manifest %3d -> %3d)  %s' % (city, b, a, before, after, why))
    CHECKS.append(dict(label='correction ' + city, got=(b, a), want=(before, after), ok=ok, note=why))

## 3. Connectivity

A network in more than one component has its time diameter fitted on a fragment. Any such network must be disclosed in the manuscript.

In [ ]:
def components(d):
    G = nx.Graph()
    G.add_nodes_from([nd['id'] for nd in d['nodes']])
    G.add_edges_from([(l['source'], l['target']) for l in d['links']])
    return sorted((len(c) for c in nx.connected_components(G)), reverse=True)

for lin in ['frozen v1', 'frozen v2']:
    print(lin + ':')
    any_split = False
    for city, d in LIN[lin].items():
        c = components(d)
        if len(c) > 1:
            any_split = True
            print('   DISCONNECTED  %-10s %s   fitted on %d of %d stations' % (city, c, c[0], sum(c)))
    if not any_split:
        print('   all single component')
    print()

## 4. Technology mix and the non steel wheel lines

In [ ]:
tech = collections.Counter(r['technology'] for r in rows)
for t, k in tech.most_common():
    print('  %3d  %s' % (k, t))

nonsteel = [r for r in rows if not r['technology'].strip().startswith('steel-wheel')]
notconv  = [r for r in rows if r['technology'].strip() != 'steel-wheel metro']
print()
print('not steel wheel            : %d lines, %d route-stations, %d cities'
      % (len(nonsteel), sum(int(r['n_stations']) for r in nonsteel), len(set(r['city'] for r in nonsteel))))
print('not conventional steel-wheel: %d lines, %d route-stations, %d cities'
      % (len(notconv), sum(int(r['n_stations']) for r in notconv), len(set(r['city'] for r in notconv))))

check('non steel wheel lines', len(nonsteel), 22)
check('non steel wheel cities', len(set(r['city'] for r in nonsteel)), 13)

In [ ]:
for grp in ['straddle monorail', 'rubber-tyred AGT / automated people mover',
            'rubber-tyred metro with central guide rail', 'medium-low-speed maglev']:
    g = [r for r in nonsteel if r['technology'].strip() == grp]
    print('%s  (%d rows)' % (grp, len(g)))
    for r in g:
        print('    %-10s %-22s %-30s n=%s' % (r['city'], r['line_local'][:22], r['line_en'][:30], r['n_stations']))
    print()
print('NOTE: the manuscript describes these at LINE level. Chongqing Line 3 is recorded as')
print('two routes (the main line and the airport branch), so seven monorail ROWS correspond')
print('to six named lines. State whichever unit the sentence uses and keep it consistent.')

## 5. The steel wheel counterfactual

Two readings are possible and they give different answers, so the manuscript must state which it means.

In [ ]:
def counterfactual(keep_fn, lineage='frozen v1'):
    drop = collections.defaultdict(set)
    for r in rows:
        if not keep_fn(r['technology']):
            drop[r['city']].add(r['route_id'])
    lost = 0; emptied = []; partial = []; frag = []
    for city, d in LIN[lineage].items():
        if city not in drop:
            continue
        dr = drop[city]; surv = set(); links = []
        for l in d['links']:
            if any(rt not in dr for rt in l['route_I_counts']):
                surv.add(l['source']); surv.add(l['target']); links.append(l)
        lost += len(d['nodes']) - len(surv)
        if not surv:
            emptied.append(city)
        else:
            if len(surv) < len(d['nodes']):
                partial.append((city, len(d['nodes']), len(surv)))
            G = nx.Graph(); G.add_nodes_from(surv)
            G.add_edges_from([(l['source'], l['target']) for l in links])
            c = sorted((len(x) for x in nx.connected_components(G)), reverse=True)
            if len(c) > 1:
                frag.append((city, c))
    return lost, emptied, partial, frag

READINGS = {
    'A  keep only "steel-wheel metro"':        lambda t: t.strip() == 'steel-wheel metro',
    'B  keep any steel-wheel technology':      lambda t: t.strip().startswith('steel-wheel'),
}
for name, fn in READINGS.items():
    lost, emptied, partial, frag = counterfactual(fn)
    tot = sum(len(d['nodes']) for d in LIN['frozen v1'].values())
    print(name)
    print('   removes %d of %d stations (%.1f%%)' % (lost, tot, 100 * lost / tot))
    print('   EMPTIED  : %s' % ', '.join(emptied))
    print('   fragments: %s' % (frag if frag else 'none'))
    print()

In [ ]:
lost, emptied, partial, frag = counterfactual(READINGS['B  keep any steel-wheel technology'])
check('counterfactual stations removed', lost, 273, 'reading B, frozen v1')
check('cities emptied', sorted(emptied), ['Macau', 'Sapporo', 'Wuhu'],
      'the manuscript must name all three, Wuhu is easy to omit')
print()
print('partially reduced:')
for city, b, a in sorted(partial, key=lambda x: a - b):
    print('   %-10s %3d -> %3d  (-%d)' % (city, b, a, b - a))

## 6. Scope, the metropolitan lines and the urban scope sample

In [ ]:
tb_all = [r for r in rows_all if 'metropolitan' in r['T3_scope']]
tb     = [r for r in rows     if 'metropolitan' in r['T3_scope']]
print('metropolitan rows in register        :', len(tb_all))
print('metropolitan rows still in the sample:', len(tb),
      '  (excluded ones leave the count:', [r['city'] + ' ' + r['line_local'] for r in tb_all if r not in tb], ')')

KR = {'Seoul', 'Busan', 'Incheon', 'Daegu'}
JP = {'Tokyo', 'Yokohama', 'Kyoto', 'Sapporo', 'Sendai', 'Fukuoka', 'Kobe'}
TW = {'Taipei', 'Kaohsiung', 'Taichung', 'Taoyuan'}
HKMO = {'Hong Kong', 'Macau'}
def ctry(c):
    return 'KR' if c in KR else 'JP' if c in JP else 'TW' if c in TW else 'HKMO' if c in HKMO else 'CN'
print('by country:', dict(collections.Counter(ctry(r['city']) for r in tb)))

check('metropolitan routes in sample', len(tb), 30)
check('of which mainland Chinese', sum(1 for r in tb if ctry(r['city']) == 'CN'), 29)
check('cities carrying one', len(set(r['city'] for r in tb)), 15)

In [ ]:
drop = collections.defaultdict(set)
for r in tb:
    drop[r['city']].add(r['route_id'])
before = after = 0; emptied = []
for city, d in LIN['frozen v1'].items():
    before += len(d['nodes'])
    if city not in drop:
        after += len(d['nodes']); continue
    dr = drop[city]; surv = set()
    for l in d['links']:
        if any(rt not in dr for rt in l['route_I_counts']):
            surv.add(l['source']); surv.add(l['target'])
    after += len(surv)
    if not surv:
        emptied.append(city)
print('urban scope sample: %d -> %d stations (-%d)' % (before, after, before - after))
print('emptied            :', emptied)
print('networks remaining :', 62 - len(emptied))

check('urban scope stations removed', before - after, 350)
check('urban scope networks', 62 - len(emptied), 58)
check('emptied cities', sorted(emptied), ['Jinhua', 'Taizhou', 'Taoyuan', 'Wenzhou'])

## 7. Table 1 indicators and the East Asian means

This reruns the study pipeline. It is the slow cell, a few minutes for 62 networks.

In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'reproduction_kit' / 'pipeline'))
sys.path.insert(0, str(ROOT / 'dataset_v2'))
from data_loader import load_city
from compute_curves import generalised_D
import importlib.util
spec = importlib.util.spec_from_file_location('rt', ROOT / 'dataset_v2' / 'reproduce_table1.py')
rt = importlib.util.module_from_spec(spec); spec.loader.exec_module(rt)

def indicators(city, ldir, pdir):
    P = load_city(ldir, pdir, city)
    D = generalised_D(P, transfer_penalty=5.0, wait_weight=2.0)
    tM = math.ceil(float(np.nanmax(D[np.isfinite(D)])))
    b = np.arange(0, tM + 1, 1.0)
    d = rt.degree_curve(D, b)
    g, th, r2 = rt.fit_eq2(b / tM, d)
    return dict(n=D.shape[0], t_M=tM, gamma=round(g, 2), theta=round(th, 3), R2=round(r2, 4),
                d30=round(float(d[min(30, len(d) - 1)]), 3),
                tau5=round(rt.tau_at(0.05, g, th), 3), tau50=round(rt.tau_at(0.50, g, th), 3),
                tau95=round(rt.tau_at(0.95, g, th), 3))
print('pipeline loaded')

In [ ]:
# Set RECOMPUTE = True to rerun all 62. Otherwise the cached table is read.
RECOMPUTE = False
cache = FROZEN / 'table1_frozen.csv'
if RECOMPUTE or not cache.exists():
    out = []
    for i, city in enumerate(sorted(LIN['frozen v1']), 1):
        r = indicators(city, FROZEN / 'L2', FROZEN / 'P2')
        out.append(dict(city=city, **{'new_' + k: v for k, v in r.items()}))
        print('[%2d/62] %-12s n=%4d t_M=%4d' % (i, city, r['n'], r['t_M']))
    T1 = out
else:
    T1 = list(csv.DictReader(open(cache, encoding='utf-8')))
    print('read cached %s, %d rows' % (cache.name, len(T1)))

f = lambda r, k: float(r[k]) if str(r[k]) not in ('', 'None') else None
print()
print('largest time diameters:')
for r in sorted(T1, key=lambda r: -f(r, 'new_t_M'))[:6]:
    print('   %-12s t_M=%4.0f  n=%4.0f' % (r['city'], f(r, 'new_t_M'), f(r, 'new_n')))

In [ ]:
vals = lambda k: [f(r, 'new_' + k) for r in T1 if f(r, 'new_' + k) is not None]
print('East Asian means on the frozen dataset')
for k in ['n', 't_M', 'gamma', 'theta', 'd30', 'tau5', 'tau50', 'tau95']:
    v = vals(k)
    print('   %-6s mean %8.3f   sd %7.3f' % (k, np.mean(v), np.std(v, ddof=1)))

check('EA mean t_M (1dp)', round(float(np.mean(vals('t_M'))), 1), 115.8)
check('EA sd t_M (1dp)',   round(float(np.std(vals('t_M'), ddof=1)), 1), 51.7)

t95 = vals('tau95'); t5 = vals('tau5')
print()
print('tau95 range [%.2f, %.2f]   tau5 range [%.2f, %.2f]' % (min(t95), max(t95), min(t5), max(t5)))
lo = min(T1, key=lambda r: f(r, 'new_tau95'))
print('smallest tau95:', lo['city'], f(lo, 'new_tau95'))
th = min(T1, key=lambda r: f(r, 'new_theta'))
print('smallest theta:', th['city'], f(th, 'new_theta'))
gm = max(T1, key=lambda r: f(r, 'new_gamma'))
print('largest gamma :', gm['city'], f(gm, 'new_gamma'))

### 7.1 Does the pipeline reproduce the submitted table on untouched cities

This is the control. If cities the freeze did not touch fail to reproduce, the difference is in the pipeline and not in the inclusion rule.

In [ ]:
CHANGED = {'Beijing', 'Daegu', 'Foshan', 'Guangzhou', 'Seoul', 'Xian'}
TOL = dict(t_M=1.0, gamma=0.005, theta=0.005, d30=0.005, tau5=0.01, tau50=0.01, tau95=0.01)
if 'old_t_M' in T1[0]:
    bad = collections.defaultdict(list)
    unchanged = [r for r in T1 if r['city'] not in CHANGED]
    for r in unchanged:
        for k, tol in TOL.items():
            o, nv = f(r, 'old_' + k), f(r, 'new_' + k)
            if o is not None and abs(nv - o) > tol:
                bad[r['city']].append((k, o, nv))
    print('%d of %d untouched cities reproduce within tolerance'
          % (len(unchanged) - len(bad), len(unchanged)))
    for c, items in bad.items():
        print('   %-12s %s' % (c, items))
    print()
    print('Deviations at the second decimal are half way rounding in the printed table.')
else:
    print('cached table has no old_* columns, rerun with RECOMPUTE = True against main.tex')

## 8. Cross city shared stations

Networks that share stations are not independent observations.

In [ ]:
def hav(a, b):
    R = 6371.0; p = math.radians
    dlat = p(b[0] - a[0]); dlon = p(b[1] - a[1])
    x = math.sin(dlat / 2) ** 2 + math.cos(p(a[0])) * math.cos(p(b[0])) * math.sin(dlon / 2) ** 2
    return 2 * R * math.asin(math.sqrt(x))

pts = {c: [(nd['name'], nd['lat'], nd['lon']) for nd in d['nodes']] for c, d in LIN['frozen v1'].items()}
cities = sorted(pts)
TOL_M = 250
pairs = collections.defaultdict(list)
for i, a in enumerate(cities):
    for b in cities[i + 1:]:
        for na, la, lo in pts[a]:
            for nb, lb, ob in pts[b]:
                if abs(la - lb) < 0.01 and abs(lo - ob) < 0.01:
                    dm = hav((la, lo), (lb, ob)) * 1000
                    if dm < TOL_M:
                        pairs[(a, b)].append((na, nb, round(dm)))
print('sharing pairs at %d m tolerance:' % TOL_M)
for (a, b), v in sorted(pairs.items(), key=lambda kv: -len(kv[1])):
    print('   %-12s %-12s %2d stations' % (a, b, len(v)))
print()
print('total shared station relationships:', sum(len(v) for v in pairs.values()))

## 9. Claim ledger

Every checked claim in one place. Anything FAIL must be fixed in the manuscript or in the dataset before submission.

In [ ]:
print('%-38s %-22s %-22s %s' % ('claim', 'computed', 'asserted', 'status'))
print('-' * 96)
for c in CHECKS:
    print('%-38s %-22s %-22s %s' % (c['label'][:38], str(c['got'])[:22], str(c['want'])[:22],
                                    'PASS' if c['ok'] else 'FAIL'))
fails = [c for c in CHECKS if not c['ok']]
print()
print('%d checks, %d pass, %d FAIL' % (len(CHECKS), len(CHECKS) - len(fails), len(fails)))
if fails:
    print()
    print('OUTSTANDING:')
    for c in fails:
        print('  - %s: computed %s, manuscript says %s. %s' % (c['label'], c['got'], c['want'], c['note']))

---

## Known outstanding items not testable here

These need a source or a decision rather than a computation.

- **Changchun Line 3** has no citation for the grade separation works. The register row says
  so. Three other files in the repository still describe the line as at grade light rail.
- **Hefei** is in two components in the v1 lineage and the seven station component sits on a
  graph route with no row in the register. Same defect class as the Beijing trams.
- **The register lacks** a section endpoint column, a retrieval date column, an approval
  instrument column, and there is no truncation table. Reviewer facing text refers to all four.
- **The regional regression** has never been run on the urban scope sample.
- **Korean designations** for Jinjeop, Hanam, Byeollae, Daegu Line 1 and the Busan Gimhae
  control case rest on secondary sources.